In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# I will try training using SKLEARN as I am more familiar with it

In [15]:
import kagglehub
path = kagglehub.competition_download('titanic')

print(path)

/kaggle/input/competitions/titanic


# Lemme try Importing the data then use onehot encodder for multiple columns


In [16]:
import pandas as pd
train_data =  pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
# print(train_data.head(5))

print(train_data.isnull().sum())
# eliminating cabin column since it has high number of  na and filling age with the mean I will avoid IMPUTTER for simplicity
# Lmme drop Names it is actually useless since we know gender
train_data = train_data.drop('Cabin',axis = 1)
train_data = train_data.drop('Name', axis = 1)
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].mean())
train_data = train_data.dropna(subset = ['Embarked'], axis = 0)

print(train_data.isnull().sum())

print(train_data)

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64
     PassengerId  Survived  Pclass     Sex        Age  SibSp  Parch  \
0              1         0       3    male  22.000000      1      0   
1              2         1       1  female  38.000000      1      0   
2              3         1       3  female  26.000000      0      0   
3              4         1       1  female  35.000000      1      0   
4              5         0       3    male  35.000000      0      0   
..           ...       ...     ...     ...        ...    ...    ...   
886          887         0       2    male  27.000000      0      0   
887     

In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_data = ['Sex', 'SibSp', 'Ticket', 'Fare', 'Embarked']
Num_data = ['PassengerId','Pclass','Age', 'Parch' ]

preprocr = ColumnTransformer(transformers = [('cat', OneHotEncoder(handle_unknown  = 'ignore',sparse_output = False),categorical_data ), ('num', 'passthrough',Num_data )])
X =  train_data.drop('Survived', axis = 1)

Y = train_data[['Survived']]

X = preprocr.fit_transform(X)
# Lemme convert it back to dataframe but not must siince I will still reconvert But I woul like to confrim something

X_feature_name =  preprocr.get_feature_names_out()

# X = pd.DataFrame(X.toarray(), columns = X_feature_name)

X = pd.DataFrame(X, columns = X_feature_name)


#print(X[:5])


print(X)



     cat__Sex_female  cat__Sex_male  cat__SibSp_0  cat__SibSp_1  cat__SibSp_2  \
0                0.0            1.0           0.0           1.0           0.0   
1                1.0            0.0           0.0           1.0           0.0   
2                1.0            0.0           1.0           0.0           0.0   
3                1.0            0.0           0.0           1.0           0.0   
4                0.0            1.0           1.0           0.0           0.0   
..               ...            ...           ...           ...           ...   
884              0.0            1.0           1.0           0.0           0.0   
885              1.0            0.0           1.0           0.0           0.0   
886              1.0            0.0           0.0           1.0           0.0   
887              0.0            1.0           1.0           0.0           0.0   
888              0.0            1.0           1.0           0.0           0.0   

     cat__SibSp_3  cat__Sib

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

#X = X
Y = Y.values.ravel()
x_train, x_test, y_train, y_test =  train_test_split(X, Y, test_size = 0.3, stratify = Y, random_state = 2)

Fdher = make_pipeline(StandardScaler(), PCA(n_components = 2), LogisticRegression(penalty = 'l2', max_iter = 10000))

Fdher.fit(x_train, y_train)
y_pred = Fdher.predict(x_test)
test_accuracy = Fdher.score(x_test,y_test)
print(f"The accuravy score is {test_accuracy}")



The accuravy score is 0.7078651685393258


# The accuracy is quiet low lemme try validation like K fold and use Grid search or Halving randomsearch CV whichever gives highest


In [35]:
from sklearn.model_selection import cross_val_score
import numpy as np


npe = cross_val_score(estimator = Fdher, X = x_train, y= y_train, cv = 10, n_jobs = -1)

print(f"The accuracy score in all cv is {npe}")

print(f"The score mean is {np.mean(npe)} and std score {np.std(npe)}")



The accuracy score in all cv is [0.73015873 0.58730159 0.69354839 0.69354839 0.77419355 0.74193548
 0.74193548 0.69354839 0.74193548 0.77419355]
The score mean is 0.7172299027137737 and std score 0.05201132121801994


# Using Grid Search

In [36]:
from sklearn.svm import SVC
import scipy.stats
from sklearn.model_selection import GridSearchCV
Fdher = make_pipeline(StandardScaler(),SVC(random_state = 9))

#Note I ignore PCA to increase accuracy
# param_range = scipy.stats.loguniform(0.0001, 1000.0)

param_range = [0.0001,0.001,0.01,1.0,10.0,100.0,1000.0]

param_grd = [{'svc__C':param_range, 'svc__kernel': ['linear'] }, {'svc__C': param_range, 'svc__gamma': param_range, 'svc__kernel': ['rbf']}]

grd = GridSearchCV(estimator = Fdher, param_grid = param_grd, cv = 10, n_jobs = -1, refit = True, scoring = 'accuracy' )

gd = grd.fit(x_train, y_train)

print(gd.best_score_)
print(f"tHE ACCURACY SCORE iS {gd.score(x_test, y_test)}")

0.8328725038402458
tHE ACCURACY SCORE iS 0.8052434456928839


In [9]:
# Halving Random Search

In [37]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

param_range = scipy.stats. loguniform(0.0001, 1000)

param_grd = [{'svc__C':param_range, 'svc__kernel': ['linear'] }, {'svc__C': param_range, 'svc__gamma': param_range, 'svc__kernel': ['rbf']}]


HRS  = HalvingRandomSearchCV(Fdher, param_distributions = param_grd  ,n_candidates = 'exhaust',resource = 'n_samples', factor = 1.5, random_state = 21,  n_jobs= -1 )

hr = HRS.fit(x_train, y_train)
print(hr.best_score_)
print(f"tHE ACCURACY SCORE iS {hr.score(x_test, y_test)}")


0.7843137254901961
tHE ACCURACY SCORE iS 0.8052434456928839


In [38]:
from sklearn.model_selection import RandomizedSearchCV


rdm = RandomizedSearchCV(estimator = Fdher, param_distributions = param_grd, scoring = 'accuracy', refit = True, n_iter = 20, cv = 10, random_state = 1, n_jobs = -1)

rs = rdm.fit(x_train, y_train)
print(rs.best_score_)
print(rs.best_params_)
print(rs.score(x_test, y_test))

0.8312852022529442
{'svc__C': np.float64(0.05971247755848463), 'svc__kernel': 'linear'}
0.8052434456928839


# Letd try confusion metrics

In [39]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, matthews_corrcoef

# Grid
Y_pred = grd.predict(x_test)
confes = confusion_matrix(y_true=y_test, y_pred=Y_pred)
print(f"The confusion matrix is:\n{confes}")

prec_val = precision_score(y_true=y_test, y_pred=Y_pred)
print(f"The precision score is {prec_val}")

recall_val = recall_score(y_true=y_test, y_pred=Y_pred)
print(f"The recall score is {recall_val}")

f1_val = f1_score(y_true=y_test, y_pred=Y_pred)
print(f"The f1 score is {f1_val}")

mcc_val = matthews_corrcoef(y_true=y_test, y_pred=Y_pred)
print(f"The Matthews correlation coefficient is {mcc_val}")

#rdm

Y_pred = HRS.predict(x_test)
confes = confusion_matrix(y_true=y_test, y_pred=Y_pred)
print(f"The confusion matrix is:\n{confes}")

prec_val = precision_score(y_true=y_test, y_pred=Y_pred)
print(f"The precision score is {prec_val}")

recall_val = recall_score(y_true=y_test, y_pred=Y_pred)
print(f"The recall score is {recall_val}")

f1_val = f1_score(y_true=y_test, y_pred=Y_pred)
print(f"The f1 score is {f1_val}")

mcc_val = matthews_corrcoef(y_true=y_test, y_pred=Y_pred)
print(f"The Matthews correlation coefficient is {mcc_val}")
Y_pred = grd.predict(x_test)
confes = confusion_matrix(y_true=y_test, y_pred=Y_pred)
print(f"The confusion matrix is:\n{confes}")

prec_val = precision_score(y_true=y_test, y_pred=Y_pred)
print(f"The precision score is {prec_val}")

recall_val = recall_score(y_true=y_test, y_pred=Y_pred)
print(f"The recall score is {recall_val}")

f1_val = f1_score(y_true=y_test, y_pred=Y_pred)
print(f"The f1 score is {f1_val}")

mcc_val = matthews_corrcoef(y_true=y_test, y_pred=Y_pred)
print(f"The Matthews correlation coefficient is {mcc_val}")

#halvingrdm




The confusion matrix is:
[[141  24]
 [ 28  74]]
The precision score is 0.7551020408163265
The recall score is 0.7254901960784313
The f1 score is 0.74
The Matthews correlation coefficient is 0.5847097761829091
The confusion matrix is:
[[141  24]
 [ 28  74]]
The precision score is 0.7551020408163265
The recall score is 0.7254901960784313
The f1 score is 0.74
The Matthews correlation coefficient is 0.5847097761829091
The confusion matrix is:
[[141  24]
 [ 28  74]]
The precision score is 0.7551020408163265
The recall score is 0.7254901960784313
The f1 score is 0.74
The Matthews correlation coefficient is 0.5847097761829091


# Time for submission I will use RandomSearchCV seems to have higher winrate

In [58]:
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
# print(test_data.isnull().sum())
test_data = test_data.drop('Cabin',axis = 1)
test_data = test_data.drop('Name', axis = 1)
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].mean())
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].mean())
test_data = test_data.dropna(subset = ['Embarked'], axis = 0)

# print(test_data.isnull().sum())

X_test_date = preprocr.transform(test_data)
Y_pred_data = rdm.predict(X_test_date)
#print(Y_pred_data)

Fina_Sub = pd.DataFrame({"passenger_id": test_data['PassengerId'], "Survived": Y_pred_data})

print(Fina_Sub)
Fina_Sub.to_csv("Submission.csv", index = False)

     passenger_id  Survived
0             892         0
1             893         1
2             894         0
3             895         0
4             896         1
..            ...       ...
413          1305         0
414          1306         1
415          1307         0
416          1308         0
417          1309         0

[418 rows x 2 columns]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
